In [1]:

import sqlite3
import uuid
from datetime import datetime

class QueryManager:
    def __init__(self, db_name="client_queries.db"):
        """Initialize database connection and create table if not exists."""
        self.conn = sqlite3.connect(db_name)
        self.cursor = self.conn.cursor()
        self.create_table()

    def create_table(self):
        """Creates the support_queries table based on your specifications."""
        sql = """
        CREATE TABLE IF NOT EXISTS support_queries (
            query_id TEXT PRIMARY KEY,
            mail_id TEXT NOT NULL,
            mobile_number TEXT,
            query_heading TEXT NOT NULL,
            query_description TEXT,
            status TEXT DEFAULT 'Open',
            query_created_time TEXT,
            query_closed_time TEXT
        )
        """
        self.cursor.execute(sql)
        self.conn.commit()

    def create_query(self, mail_id, mobile, heading, description):
        """Creates a new ticket."""
        # Generate a short unique ID (taking first 8 chars of UUID for readability)
        q_id = str(uuid.uuid4())[:8].upper()
        
        # Get current time
        created_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        sql = """
        INSERT INTO support_queries 
        (query_id, mail_id, mobile_number, query_heading, query_description, status, query_created_time) 
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """
        
        self.cursor.execute(sql, (q_id, mail_id, mobile, heading, description, "Open", created_time))
        self.conn.commit()
        print(f"\n✅ Query Created Successfully! ID: {q_id}")

    def list_queries(self, filter_status=None):
        """Lists queries. If filter_status is provided, filters by 'Open' or 'Closed'."""
        if filter_status:
            sql = "SELECT * FROM support_queries WHERE status = ?"
            self.cursor.execute(sql, (filter_status,))
        else:
            sql = "SELECT * FROM support_queries"
            self.cursor.execute(sql)
            
        rows = self.cursor.fetchall()
        
        if not rows:
            print("\n📭 No queries found.")
            return

        print(f"\n{'ID':<10} | {'Status':<8} | {'Email':<20} | {'Heading'}")
        print("-" * 60)
        for row in rows:
            # row[0]=id, row[1]=mail, row[3]=heading, row[5]=status
            print(f"{row[0]:<10} | {row[5]:<8} | {row[1]:<20} | {row[3]}")
        print("-" * 60)

    def view_query_details(self, query_id):
        """Shows full details for a specific query."""
        sql = "SELECT * FROM support_queries WHERE query_id = ?"
        self.cursor.execute(sql, (query_id,))
        row = self.cursor.fetchone()
        
        if row:
            print("\n" + "="*40)
            print(f"QUERY DETAILS: {row[0]}")
            print("="*40)
            print(f"📧 Mail ID      : {row[1]}")
            print(f"📱 Mobile       : {row[2]}")
            print(f"📌 Heading      : {row[3]}")
            print(f"📝 Description  : {row[4]}")
            print(f"📊 Status       : {row[5]}")
            print(f"🕒 Created Time : {row[6]}")
            print(f"🏁 Closed Time  : {row[7] if row[7] else 'N/A'}")
            print("="*40)
        else:
            print("\n❌ Query ID not found.")

    def close_query(self, query_id):
        """Updates status to Closed and adds closed_time."""
        # Check current status first
        self.cursor.execute("SELECT status FROM support_queries WHERE query_id = ?", (query_id,))
        result = self.cursor.fetchone()
        
        if not result:
            print("\n❌ Query ID not found.")
            return

        if result[0] == 'Closed':
            print("\n⚠️ This query is already closed.")
            return

        closed_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        sql = """
        UPDATE support_queries 
        SET status = 'Closed', query_closed_time = ? 
        WHERE query_id = ?
        """
        self.cursor.execute(sql, (closed_time, query_id))
        self.conn.commit()
        print(f"\n🔒 Query {query_id} has been closed successfully.")

    def close_connection(self):
        self.conn.close()

# --- Main Application Loop ---
def main():
    system = QueryManager()
    
    while True:
        print("\n--- CLIENT QUERY SYSTEM ---")
        print("1. Log New Query")
        print("2. View All Queries")
        print("3. View Specific Query Details")
        print("4. Close a Query")
        print("5. Exit")
        
        choice = input("Select an option (1-5): ")
        
        if choice == '1':
            email = input("Enter Email: ")
            mobile = input("Enter Mobile: ")
            heading = input("Enter Heading: ")
            desc = input("Enter Description: ")
            system.create_query(email, mobile, heading, desc)
            
        elif choice == '2':
            system.list_queries()
            
        elif choice == '3':
            q_id = input("Enter Query ID to view: ")
            system.view_query_details(q_id)
            
        elif choice == '4':
            q_id = input("Enter Query ID to close: ")
            system.close_query(q_id)
            
        elif choice == '5':
            print("Exiting...")
            system.close_connection()
            break
        else:
            print("Invalid option. Try again.")

if __name__ == "__main__":
    main()
    


--- CLIENT QUERY SYSTEM ---
1. Log New Query
2. View All Queries
3. View Specific Query Details
4. Close a Query
5. Exit


Select an option (1-5):  2



📭 No queries found.

--- CLIENT QUERY SYSTEM ---
1. Log New Query
2. View All Queries
3. View Specific Query Details
4. Close a Query
5. Exit


Select an option (1-5):  5


Exiting...


##### if choice == '1':

    email = input("Enter Email: ")
    mobile = input("Enter Mobile: ")
    heading = input("Enter Heading: ")
    desc = input("Enter Description: ")
    
    system.create_query(email, mobile, heading, desc)
